In [1]:
# !pip install pymupdf
import pymupdf
# !pip install pandas
import pandas as pd

In [2]:
import os
import json
import csv

In [35]:
jrb_edition = 6
jrb_folderpath = os.path.join("public", f"jrb-{jrb_edition}-ed")
print(jrb_folderpath) 
os.makedirs(jrb_folderpath, exist_ok=True)

public/jrb-6-ed


In [36]:
jrb_filepath = os.path.join("public", f"jrb_{jrb_edition}.pdf")
print(jrb_filepath)

song_name_page_nums = os.path.join("data_files", f"jrb_{jrb_edition}_index.tsv")
print(song_name_page_nums)

public/jrb_6.pdf
data_files/jrb_6_index.tsv


In [37]:
df = pd.read_csv(song_name_page_nums, sep='\t', names=["page_num", "song_name"], quoting=csv.QUOTE_NONE)
df['page_num'] = pd.to_numeric(df['page_num'], errors='coerce')
df_sorted = df.sort_values(by='page_num')
df_sorted

,page_num,song_name
0,10,AFRICAN FLOWER
1,11,AFRO BLUE
2,12,AFTERNOON IN PARIS
4,13,AIREGIN
3,14,AGUA DE BEBER (WATER TO DRINK)
...,...,...
394,458,YOU BROUGHT A NEW KIND OF LOVE TO ME
395,459,YOU DON'T KNOW WHAT LOVE IS
396,460,YOU TOOK ADVANTAGE OF ME
397,461,YOUNG AT HEART


In [29]:
tot = 0
for i in range(len(df_sorted)):
  current_row = df_sorted.iloc[i]
  page_scans = 1

  if i < len(df_sorted) - 1:
    next_row = df_sorted.iloc[i + 1]
    if not current_row["page_num"] + 1 == next_row["page_num"]:
      page_scans = next_row["page_num"] - current_row["page_num"]
      print(f"{current_row['page_num']:03} - {current_row['song_name']} - {page_scans}")


  tot += 1

print(tot)

004 - A FAMILY JOY - 2
020 - ANA MARIA - 2
022 - AND NOW, THE QUEEN - 0
028 - ARISE, HER EYES - 2
034 - AY, ARRIBA! - 2
038 - ICTUS - 0
062 - BRAINVILLE - 2
068 - BUTTERFLY - 2
070 - CAPTAIN MARVEL - 2
074 - CHEGA DE SAUDADE - 2
080 - CHILDREN'S SONG - 2
082 - COLORS OF CHLOE - 3
086 - COMO EN VIETNAM - 2
093 - CORAL - 0
110 - DE POIS DE AMOR O'VAZIO - 2
112 - DESAFINADO - 2
114 - DESERT AIR - 2
118 - DOIN THE PIG - 2
132 - EIDERDOWN - 2
144 - FABLES OF FAUBUS - 2
154 - FOLLOW YOUR HEART - 2
156 - FLAGS - 0
176 - GOOD EVENING MR AND MRS AMERICA - 2
182 - GROW YOUR OWN - 2
188 - HELLO, YOUNG LOVERS - 2
192 - HERZOG - 2
198 - HOTEL HELLO - 2
206 - ICARUS - 2
210 - IDA LUPINO - 2
216 - I'M ALL SMILES - 2
226 - INSIDE IN - 3
250 - JUMP MONK - 2
252 - JUNE 15th, 1967 - 2
254 - LA FIESTA - 2
264 - LITHA - 2
268 - LITURGY - 2
278 - LUSH LIFE - 2
282 - MALLET MAN - 2
284 - MAN IN THE GREEN SHIRT - 2
290 - MEVLEVIA - 2
305 - MR PC - 0
312 - MYSTERIOUS TRAVELER - 2
324 - NONSEQUENCE - 2
332 - OP

In [30]:
def extract_and_save_pages(filename, pdf_save_path, start, end):
  doc = pymupdf.open(filename)
  pages_to_keep = list(range(start, end))
  doc.select(pages_to_keep) 
  doc.save(
      pdf_save_path + ".pdf",
      garbage=3, 
      deflate=True, 
      clean=True
  )
  doc.close()

In [32]:
tot = 0
offset = 13 # this has to be computed manually.
# For:
  # jrb_6, offset is -1
  # jrb_5, offset is 13

for i in range(len(df_sorted)):
  current_row = df_sorted.iloc[i]
  page_scans = 1

  if i < len(df_sorted) - 1:
    next_row = df_sorted.iloc[i + 1]
    if not current_row["page_num"] + 1 == next_row["page_num"]:
      page_scans = next_row["page_num"] - current_row["page_num"]
      

  song_folder = os.path.join(jrb_folderpath, str(current_row["page_num"]))
  # print(song_folder)
  # print(f"{current_row['page_num']:03} - {current_row['song_name']} - {page_scans}")
  cleaned_songname = current_row['song_name'].strip().replace(" ", "_")
  # print(cleaned_songname)

  os.makedirs(song_folder, exist_ok=True)
  savename = os.path.join(song_folder, cleaned_songname)
  pdf_start_page = current_row["page_num"] - 1 + offset # subtract 1 for 0 index, add offset
  pdf_end_page = pdf_start_page + page_scans
  if pdf_start_page == pdf_end_page: # happens because 2 songs are on 1 page. 
    pdf_end_page += 1
  # print(pdf_start_page, pdf_end_page, cleaned_songname)
  extract_and_save_pages(jrb_filepath, savename, pdf_start_page, pdf_end_page)

  tot += 1

print(tot)

430


In [38]:
# Extra util to create json files
df_sorted["edition"] = jrb_edition
save_filename = os.path.join("data_files", f"jrb_{jrb_edition}_index.json")
df_sorted.to_json(save_filename, orient="records", indent=2)